In [23]:
import pandas as pd

## Load Dataset

In [24]:
df=pd.read_csv("IMDB Dataset.csv")

In [25]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [26]:
df.shape

(50000, 2)

In [27]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [28]:
df.drop_duplicates(inplace=True)

In [29]:
df.shape

(49582, 2)

## Pre-processing

### 1. Converting to Lowercase

In [30]:
df["review"]=df["review"].str.lower()

### 2. Removing URLs

In [31]:
import re

def remove_urls(text):
    text=re.sub(r"http\S+","",text) # ( pattern, replace , string) eg- https://www.google.com
    return text

df["review"]=df["review"].apply(remove_urls)

### 3. Removing Punctuation

In [32]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text) # A-Z a-z 0-9 \ s []=> use to exclude 
    return text

df["review"]=df["review"].apply(remove_punctuations)

In [33]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 4 . Removing HTML

In [34]:
def remove_html(text):
    text = re.sub(r"<.*?>","",text) # .-> any char * -> 0, any char 
    return text

df["review"]=df["review"].apply(remove_html)

### 5. Removing the Stopwords

In [35]:
!pip install nltk

In [36]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [37]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [38]:
sample_text="i love coding in python!"
tokens=word_tokenize(sample_text)

In [39]:
tokens

['i', 'love', 'coding', 'in', 'python', '!']

In [40]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")

    return text

df["review"]=df["review"].apply(remove_stopwords)        

In [41]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [43]:
# runnng -> runn
from nltk.stem import PorterStemmer

In [44]:
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"]=df["review"].apply(stemming)

In [45]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [83]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()

df["sentiment"]=le.fit_transform(df["sentiment"])

In [84]:
y=df["sentiment"]

In [85]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [86]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)

X=tf.fit_transform(df["review"])

## Dataset & Data Loader

In [107]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [108]:
X_train.shape

(39665, 5000)

In [109]:
X_test.shape

(9917, 5000)

In [110]:
import torch
from torch.utils.data import TensorDataset,DataLoader
import numpy as np

In [111]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [112]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [113]:
train_loader=DataLoader(train_set,batch_size=64,shuffle=True)
test_loader=DataLoader(test_set,batch_size=64,shuffle=True)

## Build our RNN

In [114]:
import torch.nn as nn
import torch.optim as optimizer

In [115]:
import torch
import torch.nn as nn

class RNN(nn.Module):
    
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):

        # initial hidden state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)

        # take last time step
        out = self.fc(out[:, -1, :])

        return out

In [116]:
input_size=X_train.shape[1]

model=RNN(input_size)
criteria=nn.BCELoss()
optimizer=optimizer.Adam(model.parameters())

## Training the RNN

In [117]:
epochs=10

for epoch in range(epochs):
    model.train()

    for xb,yb in train_loader:
        optimizer.zero_grad()

        xb=xb.unsqueeze(1) # add singleton direction

        outputs=model(xb) # { batch_size ,1}

        outputs=torch.sigmoid(outputs.squeeze())

        loss=criteria(outputs,yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch +1} / {epochs} train_loss = {loss.item()}")

epoch = 1 / 10 train_loss = 0.2040620893239975
epoch = 2 / 10 train_loss = 0.14400483667850494
epoch = 3 / 10 train_loss = 0.3737580478191376
epoch = 4 / 10 train_loss = 0.3034514784812927
epoch = 5 / 10 train_loss = 0.17839114367961884
epoch = 6 / 10 train_loss = 0.16222716867923737
epoch = 7 / 10 train_loss = 0.20779955387115479
epoch = 8 / 10 train_loss = 0.25406700372695923
epoch = 9 / 10 train_loss = 0.3028602600097656
epoch = 10 / 10 train_loss = 0.49425116181373596


## Evalate Model

In [121]:
model.eval()
with torch.no_grad():
    corr_vals=0
    tot_vals=0

    for xb,yb in test_loader:
        xb=xb.unsqueeze(1)

        outputs=model(xb)
        predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()

        tot_vals +=yb.size(0)
        corr_vals +=(predicted==yb).sum().item()

    print(f"accuracy :{corr_vals/tot_vals *100}")

accuracy :85.72148835333266
